# V7P3R Chess Engine - Codebase refactor
Transforming v18.3.1 into the new v7p3r standard engine. Perform cleanup and restructuring of functions, eliminate redundancy and scattered code.

## Why this exists...
This document acts as a mapping from individual engine functions to profiling variables to represent the engines decision state during a logging snapshot.

The intention is to create a modified version of v7p3r v18.3 as 18.3.1 which includes profiling and logging functionality that will report on the engines individual decision points for further analysis.

The following evaluation profile has been derived from codebase analysis of the v18.3 src code for v7p3r chess engine.

Note: After review many of the functions listed were discovered to be placeholders. Much of the v7p3r codebase may not be active during play.

## Execution Plan
1. Verification of evaluation profiling list. Must include all active v7p3r functions that impact move decisioning.
2. Extract existing evaluation profiling functions from throughout v7p3r codebase and concatinate all functions into singular v7p3r evaluators .py file. This one file will be responsible for all v7p3r positional evaluation code.
3. Extract all engine workflow and uci control functionality into singular v7p3r engine .py file. This one file will be responsible for engine control, communication, and workflow process.
4. Integrate variable logging outputs into the engine in a passive way that allows for analysis of deep dive move decision components.
5. Set up evaluation profiler to store profiling data to a big query dataset.
6. Test move selection for alignment with original v18.3 engine.
7. Choose a selection of critical positions from historical games played by v7p3r v18.3 and replay the position with v18.3.1.
8. Review move selection and evaluation profiling data across v7p3r evaluation heuristics.
9. Summarize findings and compile next gen evaluation and performance report.
10. Feedforward data and findings into the v20 ai based engine (ai based move ordering and evaluation for v7p3r)

In [ ]:
# Evaluation Profiling
eval_profile_results = {performance: {time_perf: {}, code_perf: {}, function_perf: [{}]}, ordering: {}, position: {}, fast: {}, modular: {}, bitboard: {}, safety: {}}
epr = eval_profile_results

# Move Decision Profiling Performance Metrics
epr.performance.time_perf.eval_time # total evaluation time
epr.performance.time_perf.reserved_time # total time management calculation (reserved time for move)
epr.performance.time_perf.move_time # total time for move
epr.performance.time_perf.tempo_gain # reserved time - move time
epr.performance.code_perf.function_calls # total count of function calls for this position
epr.performance.code_perf.node_count # total count of nodes explored in the move tree
epr.performance.code_perf.nodes_per_second # nodes per second
epr.performance.code_perf.depth_reached # depth reached during search
epr.performance.code_perf.cutoff_count # beta cutoff count for pruned branches
epr.performance.function_perf.function_call_id # unique id for this code performance record
epr.performance.function_perf.function_name # function being called
epr.performance.function_perf.function_runtime # runtime data about specified function

# Move Ordering Evaluator Function Results
epr.ordering.ordered_moves = V7P3REngine._order_moves_advanced()
epr.ordering.transposition_moves = tt_moves[] # transposition table lookup (class TranspositionEntry)
epr.ordering.capture_moves = captures[] # mvv-lva algorithm
epr.ordering.check_moves = checks[] # board.gives_check()
epr.ordering.killer_moves = killers[] # killer move lookup (class KillerMoves)
epr.ordering.tactical_moves = tactical_moves[] # V7P3rBitboardEvaluator.detect_bitboard_tactics()
epr.ordering.quiet_moves = quiet_moves[] # Tactical bonus < 20

# Positional Evaluator Function Results
epr.position.current_material = PositionContextCalculator._calculate_material()
epr.position.piece_info = PositionContextCalculator._calculate_piece_inventory()
epr.position.game_phase  = PositionContextCalculator._determine_game_phase()
epr.position.tactical_flags  = PositionContextCalculator._detect_tactical_flags()

# Fast Evaluator Function Results
epr.fast.combined_score = V7P3RFastEvaluator.evaluate() # Perspective based eval
epr.fast.material_score = V7P3RFastEvaluator.evaluate_material() # Material count
epr.fast.pst_score = V7P3RFastEvaluator.evaluate_pst() # Direct square indexed scores, replaced V7P3RFastEvaluator._get_piece_square_value() for 30-40% performance gains
epr.fast.opening_phase = V7P3RFastEvaluator._is_opening() # Boolean for opening phase
epr.fast.endgame_phase = V7P3RFastEvaluator._is_endgame() # Boolean for endgame phase
epr.fast.strategic_bonus = V7P3RFastEvaluator.evaluate_strategic() # Determine game phase scoring needs
epr.fast.middlegame_bonus = V7P3RFastEvaluator._calculate_middlegame_bonuses() # Calculates middlegame bonuses, Rooks on open/semi-open files, King safety - pawn shield, Pawn structure (passed pawns, doubled pawns)

# Modular Evaluator Function Results
# function name - description, cost, criticality, phases/skip, pieces
epr.modular.material_counter = ModularEvaluator._evaluate_material() # Basic material counting (P=100, N=320, B=330, R=500, Q=900), NEGLIGIBLE, ESSENTIAL, All phases
epr.modular.piece_square_tables = ModularEvaluator._evaluate_pst() # Positional bonuses for piece placement (PST), NEGLIGIBLE, ESSENTIAL, All phases
epr.modular.hanging_pieces = ModularEvaluator._evaluate_hanging_pieces() # Detect undefended pieces (captures without recapture), MEDIUM, ESSENTIAL, All Phases, KEEP
apr.modular.capture_priority = ModularEvaluator._evaluate_captures() # Prioritize recaptures and material-winning captures, LOW, ESSENTIAL, KEEP
epr.modular.check_threats = ModularEvaluator._evaluate_checks() # Evaluate check-giving moves and mate threats, MEDIUM, IMPORTANT, KEEP
# Placeholder epr.modular.pins_forks_skewers = ModularEvaluator._evaluate_tactical_patterns() # Tactical pattern detection (pins, forks, discovered attacks), MEDIUM, IMPORTANT, KEEP
epr.modular.king_safety_basic = ModularEvaluator._evaluate_king_safety_basic() # Pawn shield and basic king exposure, LOW, ESSENTIAL, OPENING/MIDDLEGAME_COMPLEX/MIDDLEGAME_SIMPLE, SKIP
epr.modular.king_safety_complex = ModularEvaluator._evaluate_king_safety_complex() # Attack patterns, tropism, storm detection, HIGH, IMPORTANT, MIDDLEGAME_COMPLEX, SKIP
epr.modular.king_centralization = ModularEvaluator.king_centralization() # King activity bonus in endgame, LOW, IMPORTANT, ENDGAME_COMPLEX/ENDGAME_SIMPLE
# Placeholder epr.modular.passed_pawns = ModularEvaluator._evaluate_passed_pawns() # Passed pawn bonuses (distance to promotion, king proximity), MEDIUM, IMPORTANT, SKIP, PAWN
# Placeholder epr.modular.doubled_pawns = ModularEvaluator._evaluate_doubled_pawns() # Penalty for doubled/tripled pawns, LOW, SITUATIONAL, SKIP, PAWN
# Placeholder epr.modular.isolated_pawns = ModularEvaluator._evaluate_isolated_pawns() # Penalty for isolated pawns (no friendly pawns on adjacent files), LOW, SITUATIONAL, SKIP, PAWN
# Placeholder epr.modular.backward_pawns = ModularEvaluator._evaluate_backward_pawns() # Penalty for backward pawns (cannot advance safely), MEDIUM, OPTIONAL, SKIP PAWN
# Placeholder epr.modular.pawn_chains = ModularEvaluator._evaluate_pawn_chains() # Bonus for connected pawn chains, LOW, SITUATIONAL, SKIP, PAWN
epr.modular.bishop_pair = ModularEvaluator._evaluate_bishop_pair() # Bonus for having both bishops (powerful in open positions), NEGLIGIBLE, SITUATIONAL, SKIP BISHOP
# Placeholder epr.modular.knight_outposts = ModularEvaluator._evaluate_knight_outposts() # Bonus for knights on strong outpost squares, LOW, OPTIONAL, MIDDLEGAME_COMPLEX/MIDDLEGAME_SIMPLE, SKIP, KNIGHT
# Placeholder epr.modular.rook_on_7th = ModularEvaluator._evaluate_rook_seventh() # Bonus for rook on 7th rank (attacking enemy pawns), LOW, SITUATIONAL, SKIP, ROOK
# Placeholder epr.modular.rook_on_open_file = ModularEvaluator._evaluate_rook_files() # Bonus for rook on open/semi-open file, LOW, SITUATIONAL, SKIP, ROOK
# Placeholder epr.modular.queen_mobility = ModularEvaluator._evaluate_queen_activity() # Queen activity and mobility evaluation, MEDIUM, IMPORTANT, .MIDDLEGAME_COMPLEX/MIDDLEGAME_SIMPLE, SKIP, QUEEN
epr.modular.piece_mobility = ModularEvaluator._evaluate_mobility() # Count legal moves for all pieces (slow, accurate), HIGH, IMPORTANT, MIDDLEGAME_COMPLEX, SKIP
epr.modular.piece_activity = ModularEvaluator.piece_activity() # Simplified mobility (attacked squares, no move gen), MEDIUM, SITUATIONAL, SKIP
# Placeholder epr.modular.center_control = ModularEvaluator._evaluate_center_control() # Control of central squares (e4, d4, e5, d5), LOW, IMPORTANT, OPENING/MIDDLEGAME_COMPLEX, SKIP
# Placeholder epr.modular.space_advantage = ModularEvaluator._evaluate_space() # Territorial control (squares controlled in opponent's half), MEDIUM, OPTIONAL, MIDDLEGAME_COMPLEX, SKIP
# Placeholder epr.modular.development = ModularEvaluator._evaluate_development() # Piece development bonus (pieces off back rank), LOW, IMPORTANT, OPENING, SKIP
# Placeholder epr.modular.opposition = ModularEvaluator._evaluate_opposition() # King opposition in pawn endgames, LOW, IMPORTANT, ENDGAME_SIMPLE, PAWN
# Placeholder epr.modular.square_of_pawn = ModularEvaluator._evaluate_pawn_races() # Can king catch passed pawn? (rule of square), LOW, IMPORTANT, ENDGAME_SIMPLE/ENDGAME_COMPLEX, PAWN
# Placeholder epr.modular.endgame_tables = ModularEvaluator._evaluate_endgame_patterns() # Theoretical endgame knowledge (KQ vs K, KR vs K, etc.), LOW, ESSENTIAL, ENDGAME_SIMPLE
# Placeholder epr.modular.see_evaluation = ModularEvaluator._evaluate_exchanges() # Static Exchange Evaluation (capture sequences), HIGH, IMPORTANT, KEEP
# Placeholder epr.modular.trapped_pieces = ModularEvaluator._evaluate_trapped_pieces() # Detect pieces with no escape squares, MEDIUM, SITUATIONAL, KEEP
# Placeholder epr.modular.back_rank_threats = ModularEvaluator._evaluate_back_rank() # Back rank mate detection and prevention, LOW, IMPORTANT, KEEP, ROOK/QUEEN
# Placeholder epr.modular.move_safety_checker = ModularEvaluator._evaluate_move_safety() # Pre-move validation (hanging pieces, legality, repetition), MEDIUM, ESSENTIAL, Always check safety
# Placeholder epr.modular.repetition_detector = ModularEvaluator._evaluate_repetition() # _evaluate_repetition Avoid threefold repetition unless desperate, LOW, ESSENTIAL
epr.modular.pawn_structure_diff = _evaluate_pawn_structure()
#.modular.connected_rooks = ModularEvaluator._evaluate_connected_rooks()
#.modular.move_tempo = ModularEvaluator._evaluate_tempo()
#.modular.king_mobility = ModularEvaluator._evaluate_king_activity_endgame()
#.modular.losing_positions = ModularEvaluator._evaluate_zugzwang()

# Bitboard Evaluator Function Results
# Main Calculators
epr.bitboard.optimized_score = V7P3RScoringCalculationBitboard.calculate_score_optimized()
epr.bitboard.bitboard_tactics = V7P3RScoringCalculationBitboard._detect_bitboard_tactics()
epr.bitboard.pawn_structure = V7P3RScoringCalculationBitboard.evaluate_pawn_structure()
epr.bitboard.king_safety = V7P3RScoringCalculationBitboard.evaluate_king_safety()
# Bitboard Evaluators
epr.bitboard.knight_attacks = V7P3RBitboardEvaluator._calc_knight_attacks()
epr.bitboard.king_attacks = V7P3RBitboardEvaluator._calc_king_attacks()
epr.bitboard.w_pawn_attacks = V7P3RBitboardEvaluator._calc_white_pawn_attacks()
epr.bitboard.b_pawn_attacks = V7P3RBitboardEvaluator._calc_black_pawn_attacks()
epr.bitboard.passed_pawn_masks = V7P3RBitboardEvaluator._generate_passed_pawn_masks()
epr.bitboard.eval_score = V7P3RBitboardEvaluator.evaluate_bitboard()
epr.bitboard.passed_count = V7P3RBitboardEvaluator._count_passed_pawns()
epr.bitboard.enhanced_castle_score = V7P3RBitboardEvaluator._evaluate_enhanced_castling()
epr.bitboard.castled = V7P3RBitboardEvaluator._has_castled()
epr.bitboard.tactical_bonus = V7P3RBitboardEvaluator._evaluate_bitboard_tactics()
epr.bitboard.fork_analysis = V7P3RBitboardEvaluator._analyze_fork_bitboard()
epr.bitboard.pins_skewers = V7P3RBitboardEvaluator._analyze_pins_skewers_bitboard()
epr.bitboard.pawn_structure = V7P3RBitboardEvaluator.evaluate_pawn_structure()
epr.bitboard.passed_pawns = V7P3RBitboardEvaluator._evaluate_passed_pawns_bitboard()
epr.bitboard.isolated_pawns = V7P3RBitboardEvaluator._evaluate_isolated_pawns_bitboard()
epr.bitboard.doubled_pawns = V7P3RBitboardEvaluator._evaluate_doubled_pawns_bitboard()
epr.bitboard.backward_pawns = V7P3RBitboardEvaluator._evaluate_backward_pawns_bitboard
epr.bitboard.connected_pawns = V7P3RBitboardEvaluator._evaluate_connected_pawns_bitboard()
epr.bitboard.pawn_chains = V7P3RBitboardEvaluator._evaluate_pawn_chains_bitboard()
epr.bitboard.pawn_storms = V7P3RBitboardEvaluator._evaluate_pawn_storms_bitboard()
epr.bitboard.is_passed_pawn = V7P3RBitboardEvaluator._is_passed_pawn_bitboard()
epr.bitboard.is_isolated_pawn = V7P3RBitboardEvaluator._is_isolated_pawn_bitboard()
epr.bitboard.is_backward_pawn = V7P3RBitboardEvaluator._is_backward_pawn_bitboard()
epr.bitboard.has_pawn_support = V7P3RBitboardEvaluator._has_pawn_support_bitboard()
epr.bitboard.found_pawn_chains = V7P3RBitboardEvaluator._find_pawn_chains_bitboard()
epr.bitboard.pawn_chain_length = V7P3RBitboardEvaluator._count_chain_length_bitboard()
epr.bitboard.connected_passed_pawn = V7P3RBitboardEvaluator._has_connected_passed_pawn()
epr.bitboard.is_on_open_file = V7P3RBitboardEvaluator._is_on_open_file_bitboard()
epr.bitboard.is_on_semi_open_file = V7P3RBitboardEvaluator._is_on_semi_open_file_bitboard()
epr.bitboard.king_safety = V7P3RBitboardEvaluator.evaluate_king_safety()
epr.bitboard.material_count = V7P3RBitboardEvaluator._count_material_bitboard()
epr.bitboard.pawn_shelter = V7P3RBitboardEvaluator._evaluate_pawn_shelter_bitboard()
epr.bitboard.castling_rights = V7P3RBitboardEvaluator._evaluate_castling_rights_bitboard()
epr.bitboard.king_exposure = V7P3RBitboardEvaluator._evaluate_king_exposure_bitboard()
epr.bitboard.escape_squares = V7P3RBitboardEvaluator._evaluate_escape_squares_bitboard()
epr.bitboard.attack_zone = V7P3RBitboardEvaluator._evaluate_attack_zone_bitboard()
epr.bitboard.enemy_pawn_storms = V7P3RBitboardEvaluator._evaluate_enemy_pawn_storms_bitboard()
epr.bitboard.king_activity = V7P3RBitboardEvaluator._evaluate_king_activity_bitboard()
epr.bitboard.is_on_open_rank = V7P3RBitboardEvaluator._is_on_open_rank_bitboard()
epr.bitboard.enemy_attacks_near_king = V7P3RBitboardEvaluator._count_enemy_attacks_near_king_bitboard()
epr.bitboard.is_safe_escape_square = V7P3RBitboardEvaluator._is_safe_escape_square_bitboard()
epr.bitboard.is_square_attacked = V7P3RBitboardEvaluator._is_square_attacked_by_enemy_bitboard()
epr.bitboard.pawn_attacks_square = V7P3RBitboardEvaluator._pawn_attacks_square_bitboard()
epr.bitboard.king_zone = V7P3RBitboardEvaluator._get_king_zone_squares()
epr.bitboard.has_castled = V7P3RBitboardEvaluator._has_castled()
epr.bitboard.bishop_pair = V7P3RBitboardEvaluator._evaluate_bishop_pair()

# Move Safety Evaluator Function Results
epr.safety.move_penalty = MoveSafetyChecker.evaluate_move_safety()
epr.safety.hanging_piece_penalty = MoveSafetyChecker._check_hanging_pieces() (MoveSafetyChecker._is_piece_hanging(), MoveSafetyChecker._get_attackers())
epr.safety.capture_penalty = MoveSafetyChecker._check_immediate_captures()
epr.safety.safe_moves = MoveSafetyChecker.get_safe_moves() (MoveSafetyChecker.evaluate_move_safety())

